# Short-Rate Calibration Lab

Module: Fixed Income, Credit, and Term Structure

## Lesson summary

The Vasicek model can be calibrated through an AR(1) bridge, while the CIR
model requires extra care because volatility depends on the rate level. This
lab estimates Vasicek parameters from the real Banxico CETES 28-day rate
history, simulates exact Vasicek paths, checks the CIR Feller condition, and
compares exact-transition Vasicek scenarios with hypothetical CIR scenarios
generated by a full-truncation Euler-Maruyama scheme
{cite}`banxicoSIE2025,vasicek1977termStructure,coxIngersollRoss1985termStructure,lordKoekkoekVanDijk2010fullTruncation`.

## Learning objectives

By the end of this lesson, students should be able to:

- estimate an AR(1) representation of the short rate;
- translate AR(1) parameters into Vasicek parameters;
- simulate exact Vasicek paths from calibrated parameters;
- evaluate the CIR Feller condition;
- simulate CIR paths with a boundary-safe discretization;
- identify model risk in one-factor short-rate models.

## Prerequisites

Complete the short-rate models, curve-fitting, and rate-panel scenario lessons
first. Readers should understand AR(1) estimation, chronological samples,
simulation seeds, the Vasicek and CIR assumptions, and the difference between
a short-rate proxy and a complete discount curve. Rates are decimals and
$\Delta t$ is expressed in years.

## Python setup

In [ ]:
import numpy as np
import pandas as pd

from src.module6_visuals import (
    build_rate_history_figure,
    build_short_rate_paths_figure,
)
from src.term_structure import (
    feller_condition,
    official_mexican_rate_history,
    simulate_cir_full_truncation,
    simulate_vasicek_exact,
    vasicek_ols_calibration,
)

## Short-rate proxy

Use CETES 28-day series SF60633 from the committed Banxico snapshot as a real
Mexican short-rate proxy for classroom calibration. The helper takes the last
provider observation within each W-FRI week and never carries a value across an
empty week; it therefore does not treat forward-filled weekends as daily
information. Rates are annualized decimals in code. Following Lesson 6.3, the
CETES field is interpreted under its ACT/360 return-yield convention. It is
neither the policy rate nor TIIE, and decimal normalization does not make those
provider quotes homogeneous. The calibration treats this quoted 28-day yield
as a short-rate proxy without converting it to an instantaneous continuously
compounded rate, so the fitted parameters inherit that classroom
approximation. Redistribution rights for the committed snapshot have not been
independently verified
{cite}`banxicoSIE2025,banxicoGovSecuritiesTechnical`.

In [ ]:
rate_history = official_mexican_rate_history(
    start="2018-01-01",
    end="2026-06-05",
    frequency="weekly",
)
rate_metadata = rate_history.attrs.copy()
short_rate = rate_history["cetes_28d"].rename("cetes_28d")
short_rate.tail()

In [ ]:
cetes_source_note = (
    f"Source: {rate_metadata['source']}, CETES 28-day "
    f"{rate_metadata['series_ids']['cetes_28d']}; {rate_metadata['alignment']}; "
    f"actual sample {rate_metadata['sample_start']} to "
    f"{rate_metadata['sample_end']} ({len(short_rate)} weeks); snapshot vintage "
    f"{rate_metadata['source_vintage']}; retrieved {rate_metadata['retrieved_at']}; "
    f"data mode {rate_metadata['data_mode']}. CETES 28-day is a short-rate proxy, "
    "not the overnight policy rate or a zero curve. Redistribution rights for "
    "the committed snapshot have not been independently verified."
)
build_rate_history_figure(
    rate_history[["cetes_28d"]],
    source_note=cetes_source_note,
)

The series exhibits persistent multi-year tightening and easing cycles rather
than rapid reversion around one fixed mean. The weekly grid is the key
exception to the source file's calendar index: it preserves a regular model
clock without inventing 252 independent CETES observations per year. The
proxy remains a quoted 28-day instrument rate, so liquidity, auction timing,
and term-premium variation are absorbed into the fitted one-factor process.

## Vasicek calibration through AR(1)

The historical calibration is interpreted under the physical measure
$\mathbb P$ for scenario analysis. The discrete bridge is:

$$
r_{t+\Delta t} = c + \beta r_t + \epsilon_t.
$$

The continuous-time parameters are:

$$
\kappa = -\frac{\ln(\beta)}{\Delta t},
\qquad
\theta = \frac{c}{1-\beta},
\qquad
\sigma =
\sqrt{\frac{2\kappa\sigma_\epsilon^2}{1-\beta^2}}.
$$

These expressions follow from the exact Vasicek transition over one weekly
step $\Delta t=1/52$ {cite}`vasicek1977termStructure`:

$$
r_{t+\Delta t}\mid r_t
\sim \mathcal N\!\left(
\theta+(r_t-\theta)e^{-\kappa\Delta t},
\frac{\sigma^2}{2\kappa}
\left(1-e^{-2\kappa\Delta t}\right)
\right),
$$

so $\beta=e^{-\kappa\Delta t}$ and a valid mean-reverting bridge requires
$0<\beta<1$. The helper rejects estimates outside that domain instead of
clipping them into a valid-looking calibration.

In [ ]:
WEEKLY_DT = 1 / 52
calibration = vasicek_ols_calibration(short_rate, dt=WEEKLY_DT)
pd.Series(calibration)

## Calibration diagnostics and sample sensitivity

Residual autocorrelation tests whether the AR(1) bridge leaves linear weekly
dependence behind. The split-sample comparison is an instability diagnostic,
not a selection rule: each subsample spans a different monetary-rate regime.

In [ ]:
calibration_frame = pd.DataFrame(
    {
        "rate": short_rate,
        "lagged_rate": short_rate.shift(1),
    }
).dropna()
calibration_frame["fitted_rate"] = (
    calibration["c"] + calibration["beta"] * calibration_frame["lagged_rate"]
)
calibration_frame["residual"] = (
    calibration_frame["rate"] - calibration_frame["fitted_rate"]
)

calibration_diagnostics = pd.Series(
    {
        "actual_start": short_rate.index.min().date().isoformat(),
        "actual_end": short_rate.index.max().date().isoformat(),
        "weekly_observations": len(short_rate),
        "delta_t_years": WEEKLY_DT,
        "frequency": rate_metadata["frequency"],
        "calendar": rate_metadata["calendar"],
        "alignment": rate_metadata["alignment"],
        "series_id": rate_metadata["series_ids"]["cetes_28d"],
        "snapshot_vintage": rate_metadata["source_vintage"],
        "retrieved_at": rate_metadata["retrieved_at"],
        "residual_mean_decimal": calibration_frame["residual"].mean(),
        "residual_std_decimal": calibration_frame["residual"].std(ddof=2),
        "residual_autocorrelation_lag_1": calibration_frame["residual"].autocorr(1),
        "rmse_basis_points": float(
            np.sqrt(np.mean(calibration_frame["residual"] ** 2)) * 10_000
        ),
    }
)
calibration_diagnostics

In [ ]:
subsample_calibrations = pd.DataFrame(
    {
        "2018-01-05 to 2021-12-31": vasicek_ols_calibration(
            short_rate.loc[:"2021-12-31"],
            dt=WEEKLY_DT,
        ),
        "2022-01-07 to 2026-06-05": vasicek_ols_calibration(
            short_rate.loc["2022-01-07":],
            dt=WEEKLY_DT,
        ),
    }
).T[["beta", "kappa", "theta", "sigma", "residual_std"]]
subsample_calibrations

## Vasicek path simulation

In [ ]:
vasicek_paths = simulate_vasicek_exact(
    r0=float(short_rate.iloc[-1]),
    kappa=calibration["kappa"],
    theta=calibration["theta"],
    sigma=calibration["sigma"],
    years=3,
    steps_per_year=52,
    paths=200,
    seed=2026,
)

## CIR Feller condition

The CIR model is {cite}`coxIngersollRoss1985termStructure`:

$$
dr_t = \kappa(\theta-r_t)dt + \sigma\sqrt{r_t}dW_t.
$$

The Feller condition is:

$$
2\kappa\theta \geq \sigma^2.
$$

In [ ]:
cir_params = {
    "r0": float(short_rate.iloc[-1]),
    "kappa": 0.75,
    "theta": 0.075,
    "sigma": 0.10,
}

pd.Series(
    {
        **cir_params,
        "feller_condition_satisfied": feller_condition(
            cir_params["kappa"],
            cir_params["theta"],
            cir_params["sigma"],
        ),
    }
)

## CIR Full Truncation Euler-Maruyama

The following CIR parameters are hypothetical and are **not** calibrated to
the CETES series. They share only the latest starting rate with the Vasicek
exercise. Full truncation advances a potentially negative auxiliary state
$\widetilde r_n$, uses $\widetilde r_n^+$ inside drift and diffusion, and
reports the non-negative value $r_n^{\mathrm{FT}}=\widetilde r_n^+$
{cite}`lordKoekkoekVanDijk2010fullTruncation`:

$$
\widetilde r_{n+1}
= \widetilde r_n
+ \kappa(\theta-\widetilde r_n^+)\Delta t
+ \sigma\sqrt{\widetilde r_n^+}\sqrt{\Delta t}Z_{n+1}.
$$

The untruncated auxiliary update feeds the next step; replacing it with the
reported non-negative value would define a different absorbing-zero scheme.

In [ ]:
cir_paths = simulate_cir_full_truncation(
    r0=cir_params["r0"],
    kappa=cir_params["kappa"],
    theta=cir_params["theta"],
    sigma=cir_params["sigma"],
    years=3,
    steps_per_year=52,
    paths=200,
    seed=2027,
)

## Distribution comparison

In [ ]:
simulation_source_note = (
    "Vasicek: OLS exact-transition bridge calibrated to weekly "
    f"{rate_metadata['source']} CETES 28-day "
    f"{rate_metadata['series_ids']['cetes_28d']}, "
    f"{rate_metadata['sample_start']} to {rate_metadata['sample_end']}, "
    f"snapshot vintage {rate_metadata['source_vintage']}, retrieved "
    f"{rate_metadata['retrieved_at']}; seed 2026. CIR: author-created parameters, "
    "not calibrated; seed 2027. Both: 3-year horizon, 52 steps/year, 200 paths; "
    "annual decimal rates displayed as percentages. Snapshot redistribution "
    "rights have not been independently verified."
)
build_short_rate_paths_figure(
    vasicek_paths,
    cir_paths,
    vasicek_theta=calibration["theta"],
    cir_theta=cir_params["theta"],
    source_note=simulation_source_note,
)

The calibrated Vasicek median moves toward its full-sample long-run estimate,
while the hypothetical CIR median moves toward 7.5%. The exception is that
the two bands are not alternative fits to the same data: only Vasicek is
estimated, and CIR dispersion reflects chosen parameters plus discretization.
The 5th-to-95th percentile bands show conditional simulation dispersion with
fixed parameters; they exclude parameter uncertainty, risk premia, and curve-fit
error.

In [ ]:
summary = pd.DataFrame(
    {
        "vasicek_final": vasicek_paths.iloc[-1],
        "cir_final": cir_paths.iloc[-1],
    }
).describe(percentiles=[0.05, 0.50, 0.95])

summary

## Model risk checklist

| Risk | Diagnostic question |
| --- | --- |
| Parameter uncertainty | Are estimates stable across samples? |
| Calibration error | Does the model match the current curve? |
| Recalibration risk | Do parameters jump sharply when new data arrive? |
| Misspecification risk | Does the model allow behavior that is economically implausible for the use case? |

## Model limitations

- The exact AR(1) bridge can still suffer finite-sample bias, time-aggregation
  effects, and proxy measurement noise.
- The Feller condition is a parameter restriction, not a complete validation of CIR fit.
- Model-generated rate paths should be interpreted with model-risk notes before being used for valuation.
- The split-sample estimates expose parameter instability; they do not identify
  which subperiod, if any, is the correct future regime.
- A historical physical-measure fit cannot price bonds without a risk-neutral
  calibration or an explicit market-price-of-risk model.

## Handoff

Module 7 converts the price, curve, DV01, spread, and calibrated-model outputs
from this module into derivative valuation, hedging, limits, and model-risk
decisions. Pass forward units, valuation timestamp, curve source, parameter
sample, and calibration diagnostics with every reported exposure.